## Pytorch setup

In [128]:
import torch

In [129]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
print(f"Using device: {device}")

Using device: cuda


## Load dataset

In [130]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

In [131]:
housing = fetch_california_housing()
x, y = housing.data, housing.target
feature_names = housing.feature_names
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_test, y_test, test_size=0.5, random_state=42)

In [132]:
print(f"Train set: {x_train.shape}, {y_train.shape}")
print(f"Test set: {x_test.shape}, {y_test.shape}")
print(f"Validation set: {x_val.shape}, {y_val.shape}")

Train set: (16512, 8), (16512,)
Test set: (2064, 8), (2064,)
Validation set: (2064, 8), (2064,)


## Preprocessing layer

In [133]:
import numpy as np
import torch.nn as nn


class PreprocessingLayer(nn.Module):
    def __init__(self, feature_names, excluded_features=("Latitude", "Longitude"), iqr_k=1.5):
        super().__init__()
        self.feature_names = feature_names
        self.excluded_features = set(excluded_features)

        self.iqr_k = iqr_k

        excluded_idx = [i for i, name in enumerate(feature_names) if name in self.excluded_features]
        non_excluded_idx = [i for i, name in enumerate(feature_names) if name not in self.excluded_features]

        self.register_buffer("excluded_idx", torch.tensor(excluded_idx, dtype=torch.long))
        self.register_buffer("non_excluded_idx", torch.tensor(non_excluded_idx, dtype=torch.long))
        self.register_buffer("lower_bounds", torch.empty(len(non_excluded_idx)))
        self.register_buffer("upper_bounds", torch.empty(len(non_excluded_idx)))
        self.register_buffer("mean", torch.empty(len(non_excluded_idx)))
        self.register_buffer("std", torch.empty(len(non_excluded_idx)))

    def _to_tensor(self, x):
        if isinstance(x, np.ndarray):
            return torch.tensor(x, dtype=torch.float32)
        else:
            return x.float()

    def fit(self, x_train):
        x_train = self._to_tensor(x_train)

        # Get quantiles for non-excluded features to determine outlier bounds.
        x_non_excluded = x_train[:, self.non_excluded_idx]
        q1 = torch.quantile(x_non_excluded, 0.25, dim=0)
        q3 = torch.quantile(x_non_excluded, 0.75, dim=0)
        iqr = q3 - q1

        self.lower_bounds.copy_(q1 - self.iqr_k * iqr)
        self.upper_bounds.copy_(q3 + self.iqr_k * iqr)

        inlier_mask = ((x_non_excluded >= self.lower_bounds) & (x_non_excluded <= self.upper_bounds)).all(dim=1)
        x_inliers = x_train[inlier_mask]
        x_non_excluded_inliers = x_inliers[:, self.non_excluded_idx]

        self.mean.copy_(x_non_excluded_inliers.mean(dim=0))
        std = x_non_excluded_inliers.std(dim=0)

        # Avoid division by zero
        std = torch.where(std == 0, torch.ones_like(std), std)
        self.std.copy_(std)

        return self

    def transform(self, x, return_mask=False, remove_outliers=True):
        x = self._to_tensor(x)
        x_non_excluded = x[:, self.non_excluded_idx]
        inlier_mask = ((x_non_excluded >= self.lower_bounds) & (x_non_excluded <= self.upper_bounds)).all(dim=1)

        x_used = x[inlier_mask] if remove_outliers else x
        x_out = x_used.clone()

        # Standardize only non-excluded features for inliers.
        x_out[:, self.non_excluded_idx] = (x_out[:, self.non_excluded_idx] - self.mean) / self.std

        if return_mask:
            return x_out, inlier_mask
        return x_out

    def forward(self, x):
        # Keep batch size unchanged inside nn.Sequential.
        return self.transform(x, return_mask=False, remove_outliers=False)


preprocessor = PreprocessingLayer(feature_names=feature_names, excluded_features=("Latitude", "Longitude"), iqr_k=1.5)
preprocessor.fit(x_train)

# Preprocess each split
x_train_pp, train_mask = preprocessor.transform(x_train, return_mask=True, remove_outliers=True)
x_val_pp, val_mask = preprocessor.transform(x_val, return_mask=True, remove_outliers=True)
x_test_pp, test_mask = preprocessor.transform(x_test, return_mask=True, remove_outliers=True)

y_train_pp = torch.tensor(y_train, dtype=torch.float32)[train_mask]
y_val_pp = torch.tensor(y_val, dtype=torch.float32)[val_mask]
y_test_pp = torch.tensor(y_test, dtype=torch.float32)[test_mask]

train_data = list(zip(x_train_pp, y_train_pp))
val_data = list(zip(x_val_pp, y_val_pp))
test_data = list(zip(x_test_pp, y_test_pp))

print(f"Train tuples: {len(train_data)}")
print(f"Val tuples:   {len(val_data)}")
print(f"Test tuples:  {len(test_data)}")

Train tuples: 13460
Val tuples:   1698
Test tuples:  1702


## Defining the models

In [ ]:
import torch.optim as optim

input_size = len(feature_names)

# Model 1
model1 = nn.Sequential(
    PreprocessingLayer(feature_names=feature_names, excluded_features=("Latitude", "Longitude"), iqr_k=1.5),
    nn.Linear(input_size, 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, 1)
)
model1[0].fit(x_train)
optimizer1 = optim.Adam(model1.parameters(), lr=0.001)

# Model 2
model2 = nn.Sequential(
    PreprocessingLayer(feature_names=feature_names, excluded_features=("Latitude", "Longitude"), iqr_k=1.5),
    nn.Linear(input_size, 50),
    nn.ReLU(),
    nn.Linear(50, 1)
)
model2[0].fit(x_train)
optimizer2 = optim.Adam(model2.parameters(), lr=0.01)

# Model 3
model3 = nn.Sequential(
    PreprocessingLayer(feature_names=feature_names, excluded_features=("Latitude", "Longitude"), iqr_k=1.5),
    nn.Linear(input_size, 100),
    nn.ReLU(),
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, 1)
)
model3[0].fit(x_train)
optimizer3 = optim.SGD(model3.parameters(), lr=0.001)

# Loss function for all 3 models
criterion = nn.MSELoss()

## Defining the minibatches creation function

In [135]:
def create_minibatches(batch_size, x, y):
    total_data = len(x)
    indices = np.arange(total_data)

    for i in range(0, total_data, batch_size):
        batch_idx = indices[i:i + batch_size]
        x_batch = torch.stack([x[j] for j in batch_idx])
        # Targets should be float tensors with shape (batch_size, 1).
        y_batch = torch.stack([y[j] for j in batch_idx]).float().unsqueeze(1)
        yield x_batch, y_batch

## Defining the train function

In [136]:
from tqdm import tqdm

In [137]:
def train(model, train_data, val_data, optimizer, criterion, epochs, batch_size, patience=None, delta=0.005):
    # Training data
    x_data = [x for x, _ in train_data]
    y_data = [y for _, y in train_data]

    # Validation data
    x_val = [x for x, _ in val_data]
    y_val = [y for _, y in val_data]

    for epoch in range(epochs):
        # Train
        model.train()
        running_loss = 0.0

        epoch_iterator = tqdm(
            create_minibatches(batch_size, x_data, y_data),
            total=(len(train_data) + batch_size - 1) // batch_size,
            desc=f"Epoch {epoch+1}/{epochs}",
            leave=True
        )

        for inputs, labels in epoch_iterator:
            optimizer.zero_grad()
            # The dataset is using cpu, check if we are using gpu and move to gpu
            if inputs.device != device:
                inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(inputs)

        epoch_train_loss = running_loss / len(train_data)
        # Eval
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for inputs, labels in create_minibatches(batch_size, x_val, y_val):
                if inputs.device != device:
                    inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * len(inputs)
        
        epoch_val_loss = val_loss / len(val_data)

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}")

        # Early stopping
        if patience is not None:
            if epoch == 0:
                best_val_loss = epoch_val_loss
                patience_counter = 0
            else:
                if epoch_val_loss < best_val_loss - delta:
                    best_val_loss = epoch_val_loss
                    patience_counter = 0
                else:
                    patience_counter += 1

            if patience_counter >= patience:
                print(f"Early stop triggered")
                break

    return epoch_train_loss, epoch_val_loss

## Trainning the models

In [138]:
model1_trained = train(model1, train_data, val_data, optimizer1, criterion, epochs=30, batch_size=64, patience=5)

Epoch 1/30: 100%|██████████| 211/211 [00:00<00:00, 301.63it/s]


Epoch [1/30], Loss: 2.2462, Val Loss: 1.1545


Epoch 2/30: 100%|██████████| 211/211 [00:00<00:00, 360.59it/s]


Epoch [2/30], Loss: 1.1540, Val Loss: 1.1200


Epoch 3/30: 100%|██████████| 211/211 [00:00<00:00, 356.68it/s]


...

Epoch [28/30], Loss: 0.4732, Val Loss: 0.4765


Epoch 29/30: 100%|██████████| 211/211 [00:01<00:00, 189.82it/s]


Epoch [29/30], Loss: 0.4719, Val Loss: 0.4723


Epoch 30/30: 100%|██████████| 211/211 [00:01<00:00, 203.32it/s]

Epoch [30/30], Loss: 0.4695, Val Loss: 0.4711


In [139]:
model2_trained = train(model2, train_data, val_data, optimizer2, criterion, epochs=20, batch_size=86, patience=3)

Epoch 1/20: 100%|██████████| 157/157 [00:00<00:00, 195.36it/s]


Epoch [1/20], Loss: 2.6123, Val Loss: 1.0360


Epoch 2/20: 100%|██████████| 157/157 [00:00<00:00, 193.52it/s]


Epoch [2/20], Loss: 0.9411, Val Loss: 0.7208


Epoch 3/20: 100%|██████████| 157/157 [00:00<00:00, 187.13it/s]


...

Epoch [9/20], Loss: 0.5313, Val Loss: 0.5272


Epoch 10/20: 100%|██████████| 157/157 [00:00<00:00, 207.59it/s]


Epoch [10/20], Loss: 0.5242, Val Loss: 0.5546


Epoch 11/20: 100%|██████████| 157/157 [00:00<00:00, 198.99it/s]


Epoch [11/20], Loss: 0.5165, Val Loss: 0.5673
Early stop triggered


In [140]:
model3_trained = train(model3, train_data, val_data, optimizer3, criterion, epochs=35, batch_size=32, patience=7)

Epoch 1/35: 100%|██████████| 421/421 [00:01<00:00, 242.18it/s]


Epoch [1/35], Loss: 2.2167, Val Loss: 1.2641


Epoch 2/35: 100%|██████████| 421/421 [00:01<00:00, 257.86it/s]


Epoch [2/35], Loss: 1.0921, Val Loss: 1.1780


Epoch 3/35: 100%|██████████| 421/421 [00:01<00:00, 253.23it/s]


...

Epoch [26/35], Loss: 0.7814, Val Loss: 0.8317


Epoch 27/35: 100%|██████████| 421/421 [00:01<00:00, 281.54it/s]


Epoch [27/35], Loss: 0.7781, Val Loss: 0.8205


Epoch 28/35: 100%|██████████| 421/421 [00:01<00:00, 284.71it/s]


Epoch [28/35], Loss: 0.7710, Val Loss: 0.8371
Early stop triggered


## Evaluating the models

In [142]:
for model, (train_loss, val_loss) in zip(
    ["Model 1", "Model 2", "Model 3"],
    [model1_trained, model2_trained, model3_trained]
):
    print(f"{model} - Final Training Loss: {train_loss:.4f}, Final Validation Loss: {val_loss:.4f}")

Model 1 - Final Training Loss: 0.4695, Final Validation Loss: 0.4711
Model 2 - Final Training Loss: 0.5165, Final Validation Loss: 0.5673
Model 3 - Final Training Loss: 0.7710, Final Validation Loss: 0.8371


In [145]:
def evaluate_model(model, dataset, criterion, batch_size=256):
    x_data = [x for x, _ in dataset]
    y_data = [y for _, y in dataset]

    model.eval()
    total_loss = 0.0
    total_abs_error = 0.0
    total_sq_error = 0.0
    test_total = 0

    with torch.no_grad():
        for inputs, labels in create_minibatches(batch_size, x_data, y_data):
            if inputs.device != device:
                inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs).squeeze(1)
            labels = labels.squeeze(1)
            loss = criterion(outputs, labels)

            total_abs_error += torch.abs(outputs - labels).sum().item()
            total_sq_error += torch.square(outputs - labels).sum().item()
            test_total += labels.size(0)
            total_loss += loss.item() * labels.size(0)

    return {
        "test_loss": total_loss / test_total,
        "test_mae": total_abs_error / test_total,
        "test_rmse": (total_sq_error / test_total) ** 0.5,
    }


models = {
    "Model 1": model1,
    "Model 2": model2,
    "Model 3": model3,
}

results = {}
for name, model in models.items():
    metrics = evaluate_model(model, test_data, criterion, batch_size=256)
    results[name] = metrics
    print(
        f"{name} = Test Loss: {metrics['test_loss']:.4f}, "
        f"Test MAE: {metrics['test_mae']:.4f}, "
        f"Test RMSE: {metrics['test_rmse']:.4f}"
    )

# Get the best model using the lowest RMSE
best_name = min(results, key=lambda k: results[k]["test_rmse"])
best_metrics = results[best_name]

print("\nBest model:", best_name)
print(
    f"{best_name} Test Loss: {best_metrics['test_loss']:.4f}, "
    f"Test MAE: {best_metrics['test_mae']:.4f}, "
    f"Test RMSE: {best_metrics['test_rmse']:.4f}"
)

Model 1 = Test Loss: 0.4595, Test MAE: 0.4963, Test RMSE: 0.6779
Model 2 = Test Loss: 0.5445, Test MAE: 0.5166, Test RMSE: 0.7379
Model 3 = Test Loss: 0.8506, Test MAE: 0.7788, Test RMSE: 0.9223

Best model: Model 1
Model 1 Test Loss: 0.4595, Test MAE: 0.4963, Test RMSE: 0.6779
